# Конвейер учебных документов с ИИ

Ключ Polza AI вводится в интерфейсе (поле «API-ключ»). Модель текста: `openai/gpt-6-luna`, изображения: `openai/gpt-image-2.5-sunburst`, адрес: `https://polza.ai/api/v1`.

Запустите ячейки по порядку, затем загрузите список тем и образец `.docx` в открывшемся интерфейсе.

In [1]:
%pip install -q python-docx openai gradio pandas openpyxl requests pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import csv
import io
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm
from docx.table import Table, _Cell
from docx.text.paragraph import Paragraph
from docx.text.run import Run
from openai import OpenAI
from PIL import Image

BASE_URL = "https://polza.ai/api/v1"
TEXT_MODEL = "openai/gpt-6-luna"
IMAGE_MODEL = "openai/gpt-image-2.5-sunburst"
CHUNK_CHARS = 6000
RETRIES = 3
OUT = Path("output")
OUT.mkdir(exist_ok=True)
HAS_LETTERS = re.compile(r"[A-Za-zА-Яа-яЁё]")

SYSTEM = """Ты методист и преподаватель. Тебе дают пронумерованные блоки текста из образца учебного документа и новую тему.
Перепиши каждый блок под новую тему: сохрани его роль в структуре, стиль и объём (±30%).
Правила:
- верни только JSON-объект {"<номер>": "<новый текст>"} со всеми номерами из запроса, без пропусков и лишних ключей;
- названия разделов, заголовки столбцов, подписи и служебные формулировки образца оставляй как есть;
- название учебного заведения оставляй, а ФИО, даты и номера групп заменяй полем «_____»;
- код, примеры и термины адаптируй под тему, не оставляй следов исходной темы образца;
- сложность, терминология и примеры должны соответствовать указанным предмету или специальности и классу или курсу."""


def read_topics(path):
    path = Path(path)
    if path.suffix == ".xlsx":
        rows = pd.read_excel(path, header=None, dtype=str).iloc[:, 0].dropna().tolist()
    else:
        text = path.read_text(encoding="utf-8-sig")
        if path.suffix == ".csv":
            delimiter = ";" if ";" in text else ","
            rows = [r[0] for r in csv.reader(text.splitlines(), delimiter=delimiter) if r]
        else:
            rows = text.splitlines()
    topics = [r.strip() for r in rows if r.strip()]
    if topics and topics[0].lower() in {"тема", "темы", "topic", "topics"}:
        topics = topics[1:]
    return list(dict.fromkeys(topics))


def read_slots(doc):
    paragraphs = (Paragraph(p, doc._body) for p in doc.element.body.iter(qn("w:p")))
    return [p for p in paragraphs if p.text.strip()]


def set_text(paragraph, text):
    runs = [r for r in paragraph._p.iter(qn("w:r")) if r.find(qn("w:t")) is not None]
    Run(runs[0], paragraph).text = text
    for r in runs[1:]:
        parent = r.getparent()
        parent.remove(r)
        if parent.tag == qn("w:hyperlink") and not len(parent):
            parent.getparent().remove(parent)


def chunks(slots):
    batch, size = [], 0
    for item in slots:
        if batch and size + len(item[1]) > CHUNK_CHARS:
            yield batch
            batch, size = [], 0
        batch.append(item)
        size += len(item[1])
    if batch:
        yield batch


def outline(slots):
    heads = [t[:60] for _, t in slots if len(t) < 80 and not t.rstrip().endswith((".", ";", ":"))]
    return "\n".join(heads[:80])


def ask(client, system, user):
    reply = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
    )
    return reply.choices[0].message.content


def parse_json(raw):
    return json.loads(raw[raw.index("{"): raw.rindex("}") + 1])


def validate(batch, data):
    if not isinstance(data, dict):
        raise ValueError("ответ не является JSON-объектом")
    expected = {str(i) for i, _ in batch}
    if set(data) != expected:
        raise ValueError(
            f"номера блоков не совпадают: нет {sorted(expected - set(data))[:10]}, "
            f"лишние {sorted(set(data) - expected)[:10]}"
        )
    for i, text in batch:
        value = data[str(i)]
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"блок {i} пустой")
        if len(text) > 80 and not 0.4 < len(value) / len(text) < 2.5:
            raise ValueError(f"блок {i}: объём {len(value)} знаков при {len(text)} в образце")


def generate(client, slots, topic, context):
    head = f"{context}\nТема: {topic}\nЗаголовки образца:\n{outline(slots)}\nБлоки:\n"
    texts = {}
    for batch in chunks(slots):
        prompt = head + json.dumps(dict(batch), ensure_ascii=False)
        error = ""
        for _ in range(RETRIES):
            try:
                data = parse_json(ask(client, SYSTEM, prompt + error))
                validate(batch, data)
                break
            except Exception as e:
                error = f"\n\nПредыдущий ответ отклонён: {e}. Верни исправленный полный JSON."
        else:
            raise RuntimeError(error.strip())
        texts.update({int(k): v for k, v in data.items()})
    return texts


def image_prompt(client, topic, context):
    system = (
        "Сформулируй одним предложением на английском промпт для иллюстрации к учебному материалу. "
        "Без текста и надписей на картинке. Верни только промпт."
    )
    return ask(client, system, f"{context}\nТема: {topic}")


def make_image(api_key, prompt, path):
    headers = {"Authorization": f"Bearer {api_key}"}
    response = requests.post(
        f"{BASE_URL}/media", headers=headers, json={"model": IMAGE_MODEL, "input": {"prompt": prompt}}, timeout=150
    )
    response.raise_for_status()
    job = response.json()
    for _ in range(60):
        if job.get("status") == "completed":
            break
        if job.get("status") in ("failed", "cancelled"):
            raise RuntimeError(job.get("error") or "генерация изображения не удалась")
        time.sleep(5)
        job = requests.get(f"{BASE_URL}/media/{job['id']}", headers=headers, timeout=30).json()
    else:
        raise TimeoutError("изображение не получено за 5 минут")
    data = job["data"]
    url = (data[0] if isinstance(data, list) else data)["url"]
    Image.open(io.BytesIO(requests.get(url, timeout=60).content)).convert("RGB").save(path)
    return str(path)


def insert_image(anchor, image_path):
    element = OxmlElement("w:p")
    anchor._p.addnext(element)
    paragraph = Paragraph(element, anchor._parent)
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    paragraph.add_run().add_picture(image_path, width=Cm(12))


def add_paragraph(container, source, text):
    style = source.style.name
    if style.startswith("Heading") and hasattr(container, "add_heading"):
        return container.add_heading(text, level=int(style[-1]) if style[-1].isdigit() else 1)
    in_list = source._p.pPr is not None and source._p.pPr.numPr is not None
    paragraph = container.add_paragraph(text, "List Bullet" if in_list else None)
    if source.runs and len(text) < 150 and all(r.bold for r in source.runs):
        for run in paragraph.runs:
            run.bold = True
    return paragraph


def fill(container, element, src, index, texts, collected=None):
    for child in element.iterchildren():
        if child.tag == qn("w:p"):
            source = Paragraph(child, src._body)
            text = texts.get(index[child], source.text) if child in index else source.text
            if text.strip():
                paragraph = add_paragraph(container, source, text.strip())
                if collected is not None:
                    collected.append(paragraph)
        elif child.tag == qn("w:tbl"):
            table = Table(child, src._body)
            new = container.add_table(rows=len(table.rows), cols=max(len(r.cells) for r in table.rows))
            new.style = "Table Grid"
            seen = set()
            for r, row in enumerate(table.rows):
                for c, cell in enumerate(row.cells):
                    if cell._tc not in seen:
                        seen.add(cell._tc)
                        fill(new.cell(r, c), cell._tc, src, index, texts)
    if isinstance(container, _Cell) and len(container.paragraphs) > 1 and not container.paragraphs[0].text:
        first = container.paragraphs[0]._p
        first.getparent().remove(first)


def build(sample, keep_format, texts, image_path, path):
    src = Document(sample)
    slots = read_slots(src)
    if keep_format:
        doc = src
        for i, paragraph in enumerate(slots):
            if i in texts:
                set_text(paragraph, texts[i])
        top = [p for p in slots if p._p.getparent().tag == qn("w:body")]
    else:
        doc, top = Document(), []
        index = {p._p: i for i, p in enumerate(slots)}
        fill(doc, src.element.body, src, index, texts, top)
    if image_path and top:
        insert_image(next((p for p in top if len(p.text) > 200), top[0]), image_path)
    doc.save(path)

In [4]:
import gradio as gr

KEEP, DROP = "Соблюдать форматирование образца", "Не соблюдать форматирование"
IMG_ON, IMG_OFF = "Добавлять изображение по теме", "Не добавлять изображение"
SCHOOL = [
    "Математика", "Алгебра", "Геометрия", "Русский язык", "Литература", "Информатика", "Физика",
    "Химия", "Биология", "География", "История", "Обществознание", "Английский язык",
]
COLLEGE = [
    "09.02.07 Информационные системы и программирование",
    "09.02.01 Компьютерные системы и комплексы",
    "09.02.06 Сетевое и системное администрирование",
    "10.02.05 Обеспечение информационной безопасности автоматизированных систем",
    "15.02.16 Технология машиностроения",
    "38.02.01 Экономика и бухгалтерский учёт (по отраслям)",
    "40.02.04 Юриспруденция",
    "44.02.02 Преподавание в начальных классах",
    "43.02.15 Поварское и кондитерское дело",
    "08.02.01 Строительство и эксплуатация зданий и сооружений",
    "34.02.01 Сестринское дело",
]
CLASSES = [f"{n} класс" for n in range(1, 12)]
COURSES = [f"{n} курс" for n in range(1, 5)]
COLUMNS = ["Тема", "Статус", "Файл", "Примечание"]


def path_of(file):
    return getattr(file, "name", file)


def result_files(state):
    return [d["path"] for d in state["docs"].values()] + [str(OUT / "log.csv")]


def run(key, topics_file, sample_file, keep, level, subject, course, with_image):
    if not (key and topics_file and sample_file):
        raise gr.Error("Укажите API-ключ, файл с темами и образец документа")
    sample, topics = path_of(sample_file), read_topics(path_of(topics_file))
    client = OpenAI(base_url=BASE_URL, api_key=key.strip())
    slots = [(i, p.text) for i, p in enumerate(read_slots(Document(sample))) if HAS_LETTERS.search(p.text)]
    school = level == "Школа"
    context = f"{level}. {'Предмет' if school else 'Специальность'}: {subject}. {'Класс' if school else 'Курс'}: {course}."
    state = {"sample": sample, "keep": keep == KEEP, "docs": {}}
    log = []
    for topic in topics:
        log.append([topic, "в работе", "", ""])
        yield log, state, gr.update(), None
        try:
            texts = generate(client, slots, topic, context)
            stem = re.sub(r'[\\/:*?"<>|\r\n]+', " ", topic).strip()[:80]
            path, image, note = OUT / f"{stem}.docx", None, ""
            if with_image == IMG_ON:
                try:
                    image = make_image(key.strip(), image_prompt(client, topic, context), OUT / f"{stem}.png")
                except Exception as e:
                    note = f"без изображения: {e}"
            build(sample, state["keep"], texts, image, path)
            state["docs"][topic] = {"texts": texts, "image": image, "path": str(path)}
            log[-1] = [topic, "готово", path.name, note]
        except Exception as e:
            log[-1] = [topic, "ошибка", "", str(e)]
    pd.DataFrame(log, columns=COLUMNS).to_csv(OUT / "log.csv", index=False, encoding="utf-8-sig")
    done = list(state["docs"])
    yield log, state, gr.update(choices=done, value=done[0] if done else None), result_files(state)


def on_level(level):
    school = level == "Школа"
    options, grades = (SCHOOL, CLASSES) if school else (COLLEGE, COURSES)
    return (
        gr.update(choices=options, value=options[0], label="Предмет" if school else "Специальность"),
        gr.update(choices=grades, value=grades[1], label="Класс" if school else "Курс"),
    )


def show(topic, state):
    doc = state["docs"].get(topic)
    if not doc:
        return "", None
    return "\n".join(f"[{i}] {t.replace(chr(10), ' ')}" for i, t in doc["texts"].items()), doc["image"]


def apply_edit(topic, text, state):
    if topic not in state["docs"]:
        raise gr.Error("Выберите тему")
    doc = state["docs"][topic]
    for line in text.splitlines():
        match = re.match(r"\[(\d+)\] (.*)", line)
        if match and int(match[1]) in doc["texts"]:
            doc["texts"][int(match[1])] = match[2]
    build(state["sample"], state["keep"], doc["texts"], doc["image"], doc["path"])
    gr.Info("Файл пересобран")
    return state, result_files(state)


with gr.Blocks(title="Конвейер учебных материалов") as demo:
    state = gr.State({"docs": {}})
    key = gr.Textbox(label="API-ключ Polza AI", type="password")
    with gr.Row():
        topics_file = gr.File(label="Список тем (.txt, .csv, .xlsx)", file_types=[".txt", ".csv", ".xlsx"])
        sample_file = gr.File(label="Образец документа (.docx)", file_types=[".docx"])
    with gr.Row():
        level = gr.Radio(["Школа", "Колледж"], value="Колледж", label="Уровень")
        subject = gr.Dropdown(COLLEGE, value=COLLEGE[0], label="Специальность", allow_custom_value=True)
        course = gr.Dropdown(COURSES, value=COURSES[1], label="Курс", allow_custom_value=True)
    with gr.Row():
        keep = gr.Radio([KEEP, DROP], value=KEEP, label="Форматирование")
        with_image = gr.Radio([IMG_ON, IMG_OFF], value=IMG_ON, label="Изображения")
    start = gr.Button("Запустить", variant="primary")
    status = gr.Dataframe(headers=COLUMNS, label="Статус обработки", interactive=False, wrap=True)
    files = gr.File(label="Готовые файлы", file_count="multiple")
    topic = gr.Dropdown(label="Предпросмотр и правка: тема")
    with gr.Row():
        editor = gr.Textbox(label="Текст документа: правьте текст после «[номер] »", lines=18, max_lines=30)
        picture = gr.Image(label="Изображение", interactive=False)
    apply = gr.Button("Применить правку и пересобрать файл")

    level.change(on_level, level, [subject, course])
    start.click(run, [key, topics_file, sample_file, keep, level, subject, course, with_image], [status, state, topic, files])
    topic.change(show, [topic, state], [editor, picture])
    apply.click(apply_edit, [topic, editor, state], [state, files])

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\gradio\blocks.py", line 1975, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^